# Elmatrix Studio v5 — story-driven videos

Studio v5 plans scenes from your script, uses niche-specific visual directions, prefers neural narration, and burns captions into the video. Optional AI-generated or licensed stills can be included in your job ZIP as `assets/scene-01.png`, `assets/scene-02.jpg`, etc. No paid image API is called and no stock footage is downloaded. Check rights for every image.


In [ ]:
import subprocess, sys
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','espeak-ng','fonts-dejavu-core'], check=True)
subprocess.run([sys.executable,'-m','pip','install','Pillow==12.3.0','edge-tts'], check=True)
print('Studio v5 dependencies ready.')


In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile, zipfile
from google.colab import files
from IPython.display import Video, display

os.environ['ELMATRIX_VOICE_ENGINE'] = 'auto'
os.environ['ELMATRIX_NEURAL_VOICE'] = 'en-ZA-LeahNeural'
os.environ['ELMATRIX_VOICE_RATE'] = '-8%'
os.environ['ELMATRIX_VOICE_PITCH'] = '+0Hz'

url='https://raw.githubusercontent.com/braze222/ish-toolkit/feature/video-pilot/video-pilot/renderer_v5.py'
work=Path(tempfile.mkdtemp(prefix='elmatrix-studio-v5-')); renderer=work/'renderer_v5.py'
subprocess.run(['curl','-fsSL',url,'-o',str(renderer)],check=True)
if renderer.stat().st_size<12000: raise RuntimeError('Renderer download appears incomplete.')
print('Upload one job ZIP with project.json. Optional images: assets/scene-01.png, assets/scene-02.jpg, etc. Max ZIP 32 MiB.')
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError('Choose exactly one job ZIP.')
name,content=next(iter(uploaded.items()))
if not name.lower().endswith('.zip') or len(content)>32*1024*1024: raise ValueError('Expected one ZIP no larger than 32 MiB.')
job=work/'job.zip'; job.write_bytes(content); result=work/'render-result-v5.zip'
subprocess.run([sys.executable,'-I',str(renderer),str(job),str(result)],check=True,timeout=2400)
with zipfile.ZipFile(result) as z:
    if sorted(z.namelist())!=['render.json','video.mp4'] or z.testzip() is not None: raise RuntimeError('Result ZIP failed integrity checks.')
    preview=work/'preview.mp4'; preview.write_bytes(z.read('video.mp4')); print(z.read('render.json').decode())
display(Video(str(preview),embed=True)); files.download(str(result))
print('Save render-result-v5.zip to Files/xxix. Import into a NEW project directory. Review audio, captions, and image rights. Nothing was published.')
